# Unit 3 — Write ACT training

You will implement the loss and update path that changes ACT weights. Earlier units used a ready policy. Here, the network and fixed training and validation batches are supplied. Unit 5 will show how to build those batches.

Each short exercise builds one part of the training path. The final experiment compares learning rates on the same ACT data with two seeds and a fixed update budget. It is opt-in because it needs local data and can take several minutes. `Run All` does not start it.

The ACT paper uses a masked action L1 loss plus a KL penalty for its latent posterior. The KL term sums over latent values and averages over the batch. Reading: [ACT paper](https://arxiv.org/abs/2304.13705).

**PyTorch notes:** a tensor with `requires_grad=True` records operations for autograd. `backward()` fills gradients. `optimizer.step()` changes parameters. Clear gradients before each update. Call `detach()` only when you record a value for display.

This notebook contains the full implementation.


In [ ]:
from pathlib import Path
import os, sys

def find_workspace():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "manifests" / "act_references.yaml").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside this repository.")

WORKSPACE = find_workspace()
sys.path.insert(0, str(WORKSPACE / "src"))
os.environ.setdefault("HF_HUB_CACHE", str(WORKSPACE / ".local-data/hf/hub"))
os.environ.setdefault("HF_DATASETS_CACHE", str(WORKSPACE / ".local-data/hf/datasets"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / ".local-data/matplotlib"))
import torch
from torch import nn
import matplotlib.pyplot as plt
print("Workspace:", WORKSPACE)
print("PyTorch:", torch.__version__)


In [ ]:
# A small deterministic fixture. It checks shapes and padding without downloading data.
B, T, D, L = 3, 4, 2, 3
fixture = {
    "action": torch.full((B, T, D), 0.5),
    "action_is_pad": torch.tensor([[0,0,0,0],[0,0,1,1],[0,1,1,1]], dtype=torch.bool),
}
class TinyACT(nn.Module):
    def __init__(self):
        super().__init__()
        self.action_value = nn.Parameter(torch.zeros(D))
        self.mu_value = nn.Parameter(torch.ones(L) * 0.1)
    def forward(self, batch):
        pred = self.action_value.view(1,1,D).expand_as(batch["action"])
        mu = self.mu_value.unsqueeze(0).expand(batch["action"].shape[0], -1)
        return pred, (mu, torch.zeros_like(mu))


## Exercises

Reference implementations and checks.

## Exercise 1 — Masked action loss (about 15 min)

Implement `masked_action_l1(prediction, target, is_pad)`. Inputs have shapes `(B,T,D)`, `(B,T,D)`, `(B,T)`. `is_pad=True` means the timestep is padding. Return the mean absolute error across valid action values. An all-padding batch returns zero.

Formula: sum absolute error at valid times / max(valid timestep count × D, 1). The mask must not give padded predictions a gradient.

**Pointers:** Expand the time mask over the action dimension. Keep the reduction as a tensor operation.

<details><summary>More hints</summary>

Invert the bool mask, add a final axis, cast to the prediction dtype, then multiply. Clamp the denominator to one.

</details>

In [ ]:
def masked_action_l1(prediction, target, is_pad):
    if prediction.shape != target.shape or prediction.ndim != 3:
        raise ValueError("prediction and target must have the same (B,T,D) shape")
    if is_pad.shape != prediction.shape[:2]:
        raise ValueError("is_pad must have shape (B,T)")
    valid = (~is_pad.bool()).unsqueeze(-1).to(prediction.dtype)
    denominator = (valid.sum() * prediction.shape[-1]).clamp_min(1)
    return ((prediction - target).abs() * valid).sum() / denominator

In [ ]:
p = torch.zeros_like(fixture["action"], requires_grad=True)
loss = masked_action_l1(p, fixture["action"], fixture["action_is_pad"])
assert loss.shape == () and abs(loss.item() - 0.5) < 1e-6, "Expected scalar valid-step MAE of 0.5."
loss.backward()
assert torch.count_nonzero(p.grad[fixture["action_is_pad"]]) == 0, "Padded positions must have zero gradient."
all_pad = torch.ones_like(fixture["action_is_pad"])
assert masked_action_l1(p.detach(), fixture["action"], all_pad).item() == 0, "All-padding loss must be zero."
print("Masked loss:", loss.item())

In [ ]:
pred_changed = p.detach().clone()
pred_changed[fixture["action_is_pad"]] = 900
assert torch.allclose(masked_action_l1(pred_changed, fixture["action"], fixture["action_is_pad"]), loss.detach()), "Padded values must not change loss."
print("Use: this function is called by the ACT objective in Exercise 3.")

## Exercise 2 — Gaussian KL (about 15 min)

Implement the KL divergence from `N(mu, exp(logvar))` to `N(0,I)`. `mu` and `logvar` each have shape `(B,L)`. Sum over `L`, then average over `B`.

Formula per dimension: `-0.5 * (1 + logvar - mu**2 - exp(logvar))`.

**Pointers:** Take care with the order of reductions. `logvar` is the log of variance.

<details><summary>More hints</summary>

Use `torch.exp(logvar)`. Reduce with `.sum(dim=-1).mean()`.

</details>

In [ ]:
def diagonal_gaussian_kl(mu, logvar):
    if mu.shape != logvar.shape or mu.ndim != 2:
        raise ValueError("mu and logvar must have the same (B,L) shape")
    return (-0.5 * (1 + logvar - mu.square() - logvar.exp())).sum(-1).mean()

In [ ]:
mu = torch.tensor([[1.0, 0.0], [0.0, 0.0]])
logvar = torch.tensor([[0.0, 0.0], [0.0, 1.38629436]])
expected = (0.5 + 0.5 * (4.0 - 1.0 - 1.38629436)) / 2
value = diagonal_gaussian_kl(mu, logvar)
assert value.shape == () and abs(value.item() - expected) < 1e-6, "Sum over latent width, then mean over batch."
z = torch.zeros(2, 3)
assert diagonal_gaussian_kl(z, z).item() == 0, "The standard normal KL must be zero."
print("KL:", value.item())

In [ ]:
print("Use: Exercise 3 adds this value to the action loss with a KL weight.")

## Exercise 3 — One update from a loss callback (about 15 min)

Implement `optimizer_update(model, batch, optimizer, loss_fn, max_grad_norm=None)`. The callback has this contract: `loss_fn(model, batch, *, noise=None, time=None) -> (scalar_loss, metrics)`. It must keep the autograd graph until the update. Return detached Python numbers after the update.

**Method:** training mode → clear old gradients → call the loss function → backpropagate → optionally clip the gradient norm → update weights.

**Pointers:** The callback can return extra scalar metrics. The loss must be a finite scalar tensor.

<details><summary>More hints</summary>

Use `optimizer.zero_grad(set_to_none=True)`. Pass `noise` and `time` as keyword arguments, including when they are `None`. Check the loss before `backward()`. Use `.detach().cpu()` to record scalars. If you clip, check that the limit is positive and clip after `backward()`.

</details>

In [ ]:
def optimizer_update(model, batch, optimizer, loss_fn, *, noise=None, time=None, max_grad_norm=None):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss, metrics = loss_fn(model, batch, noise=noise, time=time)
    if loss.ndim != 0 or not torch.isfinite(loss):
        raise ValueError("loss callback must return one finite scalar tensor")
    loss.backward()
    if max_grad_norm is not None:
        if max_grad_norm <= 0: raise ValueError("max_grad_norm must be positive")
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
    optimizer.step()
    result = {"loss": float(loss.detach().cpu())}
    for key, value in metrics.items():
        result[key] = float(torch.as_tensor(value).detach().cpu())
    return result

In [ ]:
model = TinyACT()
opt = torch.optim.SGD(model.parameters(), lr=0.1)
def tiny_act_loss(model, batch, *, noise=None, time=None):
    predicted, (mu, logvar) = model(batch)
    action_loss = masked_action_l1(predicted, batch["action"], batch["action_is_pad"])
    kl = diagonal_gaussian_kl(mu, logvar)
    return action_loss + 0.01 * kl, {"action_l1": action_loss, "kl": kl}
before = model.action_value.detach().clone()
result = optimizer_update(model, fixture, opt, tiny_act_loss)
assert set(result) == {"loss", "action_l1", "kl"} and all(torch.isfinite(torch.tensor(v)) for v in result.values()), "Return finite scalar metrics."
assert not torch.equal(before, model.action_value.detach()), "The update must change model weights."
print(result)

In [ ]:
print("Use: optimizer_update passes the loss callback through to the actual optimizer step.")

## Exercise 4 — Fixed update loop (about 15 min)

Implement `train_loop(model, batches, optimizer, loss_fn, *, steps, max_grad_norm=None)`. Cycle through the supplied batches in order. Pass the optional gradient limit to each update. Return one metric dictionary per update. Reject an empty batch list and a negative step count.

**Pointers:** Convert the iterable to a list once. Use the modulo operator to cycle.

<details><summary>More hints</summary>

Call your `optimizer_update` each time. For step `i`, use batch `i % len(batches)`.

</details>

In [ ]:
def train_loop(model, batches, optimizer, loss_fn, *, steps, max_grad_norm=None):
    batches = list(batches)
    if not batches:
        raise ValueError("batches must contain at least one batch")
    if steps < 0:
        raise ValueError("steps must be non-negative")
    return [optimizer_update(model, batches[i % len(batches)], optimizer, loss_fn, max_grad_norm=max_grad_norm) for i in range(steps)]

In [ ]:
model = TinyACT()
opt = torch.optim.SGD(model.parameters(), lr=0.1)
history = train_loop(model, [fixture], opt, tiny_act_loss, steps=8, max_grad_norm=1.0)
assert len(history) == 8 and all("loss" in item for item in history), "Return exactly one metric row per update."
assert history[-1]["loss"] < history[0]["loss"], "Repeated updates should reduce this fixed fixture loss."
plt.plot([item["loss"] for item in history])
plt.xlabel("Update")
plt.ylabel("Fixture loss")
plt.title("Mechanics check only")
plt.show()

In [ ]:
print("Use: the same loop accepts any callback that follows the documented contract.")

## Experiment — ACT learning rates on fixed data (30–60 min)

Use the supplied preparation function to load one small, fixed training and validation set. Compare three learning rates with two seeds. Use the same model initialization for all rates within each seed. Keep the update budget fixed. Do not update on validation batches.

**Prediction:** write which rate you expect to perform best and why.

**Fixed conditions:** use the same train and validation episodes, chunk size, batch size, optimizer, KL weight, and number of updates. Record every trial, including worse or failed trials. Compare mean validation action L1 and the two per-seed results. State when the result is inconclusive.

Set `RUN_ACT_LR_EXPERIMENT = True` only after the short exercises pass and local training data are installed. This block has a fixed step limit. It does not download data or change a saved policy. Run trials one at a time. Write a short conclusion after the table.

The split is frozen. Training statistics come from training episodes only. See the metadata printed by the helper. A learner-training result is distinct from running prepared reference weights.' 

In [ ]:
RUN_ACT_LR_EXPERIMENT = False  # Explicit opt-in. Run All stays bounded.
if RUN_ACT_LR_EXPERIMENT:
    import gc
    from lerobot.policies.act import ACTPolicy
    from vla_lab.policy.act import make_act_config
    from vla_lab.teaching.live import load_act_batches, make_act_raw_model
    act_data = load_act_batches("bowl_plate", WORKSPACE, batch_size=2, chunk_size=20, train_batches=2, validation_batches=1, seed=17)
    print("Frozen batch metadata:", act_data.metadata)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    def move_batches(rows):
        return [{key: value.to(device) if isinstance(value, torch.Tensor) else value for key, value in batch.items()} for batch in rows]
    train_batches, validation_batches = move_batches(act_data.train), move_batches(act_data.validation)
    rates, seeds, update_budget = [1e-5, 3e-5, 1e-4], (7, 19), 20
    rows = []
    for seed in seeds:
        torch.manual_seed(seed)
        initial_policy = ACTPolicy(make_act_config(device="cpu", chunk_size=20))
        initial_state = {key: value.detach().cpu().clone() for key, value in initial_policy.state_dict().items()}
        del initial_policy
        for rate in rates:
            row = {"seed": seed, "learning_rate": rate, "train_curve": [], "validation_curve": [], "status": "failed"}
            policy = model = optimizer = None
            try:
                policy = ACTPolicy(make_act_config(device=str(device), chunk_size=20)).to(device)
                policy.load_state_dict(initial_state)
                model = make_act_raw_model(policy)
                optimizer = torch.optim.AdamW(policy.get_optim_params(), lr=rate, weight_decay=1e-4)
                # Reset the random stream so each rate sees the same posterior samples and dropout sequence.
                torch.manual_seed(seed + 1000)
                if device.type == "cuda": torch.cuda.manual_seed_all(seed + 1000)
                def learner_act_loss(current_model, batch, *, noise=None, time=None):
                    del noise, time
                    predicted, (mu, logvar) = current_model(batch)
                    action_loss = masked_action_l1(predicted, batch["action"], batch["action_is_pad"])
                    kl = diagonal_gaussian_kl(mu, logvar)
                    return action_loss + 10.0 * kl, {"action_l1": action_loss, "kl": kl}
                validation_updates = []
                validation_interval = 5
                for update_start in range(0, update_budget, validation_interval):
                    batch_offset = update_start % len(train_batches)
                    ordered_batches = train_batches[batch_offset:] + train_batches[:batch_offset]
                    interval_steps = min(validation_interval, update_budget - update_start)
                    step_history = train_loop(model, ordered_batches, optimizer, learner_act_loss, steps=interval_steps, max_grad_norm=10.0)
                    row["train_curve"].extend(item["action_l1"] for item in step_history)
                    model.eval()
                    with torch.no_grad():
                        val_errors = []
                        for batch in validation_batches:
                            predicted, _ = model(batch)  # ACT returns no posterior statistics in evaluation mode.
                            val_errors.append(masked_action_l1(predicted, batch["action"], batch["action_is_pad"]).item())
                    row["validation_curve"].append(sum(val_errors) / len(val_errors))
                    validation_updates.append(update_start + interval_steps)
                    model.train()
                row["validation_updates"] = validation_updates
                row["status"] = "complete"
            except Exception as error:
                row["error"] = f"{type(error).__name__}: {error}"
            rows.append(row)
            print({key: value for key, value in row.items() if key not in {"train_curve", "validation_curve"}})
            del optimizer, model, policy
            gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
    for row in rows:
        if row["train_curve"]:
            plt.plot(row["train_curve"], alpha=0.5, label=f'train lr={row["learning_rate"]:g}, seed={row["seed"]}')
            plt.plot(row["validation_updates"], row["validation_curve"], linestyle="--", marker="o", label=f'val lr={row["learning_rate"]:g}, seed={row["seed"]}')
    plt.xlabel("Update (validation every 5 updates)")
    plt.ylabel("Action L1")
    plt.legend(fontsize=7, ncol=2)
    plt.show()
    for row in rows:
        print("trial curves:", row["seed"], row["learning_rate"], row["status"], "train", row["train_curve"], "validation", row["validation_curve"])
    print("Keep all completed and failed trials. Compare both curves and both seeds. This short run does not show policy success.")
else:
    print("Skipped. Set RUN_ACT_LR_EXPERIMENT=True to run the fixed two-seed ACT experiment.")

## References and limits

ACT training uses a posterior during training and a zero latent at inference. The short experiment checks whether the callback changes a real ACT model on prepared batches. It does not establish that the resulting policy succeeds in the environment. The recorded dataset is 10 Hz; the simulator controller is 20 Hz. The course preserves the current timing behavior until Unit 13 tests it explicitly.